# Regional GAA Profiling
Owner: Anje
Table: `pondolens.dev_bronze.regional_expenditure_gaa`
Baseline: 4,780 rows

**Goal:** identify population records suitable for Q3 and required Silver treatments. All queries are read-only.

**Row meaning:** A government expenditure allocation for a specific fiscal year, expenditure category, particular (department or special purpose fund), and geographical area.

## Checks
- Check counts by file/year, column types, and nulls/blanks.
- Define what one row represents; check exact duplicates and repeated business keys.
- Verify regions, years, categories, particulars, numeric amounts, units, and price basis.
- Identify totals/subtotals to avoid double counting.
- Compare sample amounts and totals with the original source.
- With Joy, check whether GAA and obligations have matching scope, periods, and regions.
- Check regional compatibility with population and GRDP.

## Findings
| Check / Issue | Affected rows / Evidence | Proposed Silver treatment |
|---|---|---|
| Dashes in expenditure_amount column| 336 rows in 2025 table, 344 rows in 2024 table| treat as NULL since no data was inputted in the original tables|
| Dpecial characters in particulars column (). The data contains both the department and the acronym of the department inside the () | 40 distinct column_value with 20 occurences each for both 2024 and 2025 tables. 39 of these values are the agency names while the other value is GAA(updated)| separate the acronym from the full agency names|
| when particulars = 'All', these are aggregate rows. Other particulars are department or fund detail rows rathen than automatically being subtotals | All rows| Keep all rows in Silver and add two classification columns. This preserves published totals for validation while allowing dashboards to select detail rows without double counting. financial_row_type: grand_total, category_total, detail AND geographic_row_type: geographic_total, nonregional, regional|
|Expenditure amount in thousand pesos|all rows| convert expenditure amount to be in thousand pesos and remove unit_amoount column|

## Assumptions to Verify
| Question | Status |
|---|---|

## Semantic Notes
- The amount is expressed in thousand Philippine pesos (₱'000).

## Conclusion
- Required Silver changes:
- Ready for Q1/Q3? Why / why not:
- Open questions:

## 1. Structure and coverage
Inspect columns, sample records, years, and geographic labels. PHILIPPINES is a national total, not a region.

In [0]:
%sql
DESCRIBE TABLE pondolens.dev_bronze.regional_expenditure_gaa;

col_name,data_type,comment
source_name,string,null
source_page,string,null
price_basis,string,null
year,string,null
category,string,null
particulars,string,null
regions,string,null
expenditure_amount,string,null
unit_amount,string,null
_rescued_data,string,null


In [0]:
%sql
SELECT *
FROM pondolens.dev_bronze.regional_expenditure_gaa
LIMIT 10;

source_name,source_page,price_basis,year,category,particulars,regions,expenditure_amount,unit_amount,_rescued_data,_drive_file_id,_source_file,_landing_file,_source_modified_at,_source_sheet,_content_hash,_header_map,_dataset_hash,_run_id,_ingested_at
03_regional_expenditure_gaa_2025.pdf,1,GAA (Updated),2025,DEPARTMENTS,All,Region I,136034435,thousand pesos,null,1wgAxpzoqU7VZtmwGC1IEi7cHvmk88MQR,03_regional_expenditure/03_regional_expenditure_gaa_2025.csv,/Volumes/pondolens/dev_bronze/government_budget/gdrive_landing/1wgAxpzoqU7VZtmwGC1IEi7cHvmk88MQR/3d392c835a11f16191582566fc20ddb49115f7e238a9e62dcb7e1c353ae51180.csv,2026-10-08T16:07:40.501Z,null,3d392c835a11f16191582566fc20ddb49115f7e238a9e62dcb7e1c353ae51180,"{""source_name"": ""source_name"", ""source_page"": ""source_page"", ""price_basis"": ""price_basis"", ""year"": ""year"", ""category"": ""category"", ""particulars"": ""particulars"", ""regions"": ""regions"", ""expenditure_amount"": ""expenditure_amount"", ""unit_amount"": ""unit_amount""}",a5d05e136597e195af5ab6d78be6d895b227af2c7ebd5b19f3c59452234c87b0,d1c85596cecb41109d7f79ad07b35905,2026-10-12T14:58:10.107Z
03_regional_expenditure_gaa_2025.pdf,1,GAA (Updated),2025,DEPARTMENTS,All,Cordillera Administrative Region,75036825,thousand pesos,null,1wgAxpzoqU7VZtmwGC1IEi7cHvmk88MQR,03_regional_expenditure/03_regional_expenditure_gaa_2025.csv,/Volumes/pondolens/dev_bronze/government_budget/gdrive_landing/1wgAxpzoqU7VZtmwGC1IEi7cHvmk88MQR/3d392c835a11f16191582566fc20ddb49115f7e238a9e62dcb7e1c353ae51180.csv,2026-10-08T16:07:40.501Z,null,3d392c835a11f16191582566fc20ddb49115f7e238a9e62dcb7e1c353ae51180,"{""source_name"": ""source_name"", ""source_page"": ""source_page"", ""price_basis"": ""price_basis"", ""year"": ""year"", ""category"": ""category"", ""particulars"": ""particulars"", ""regions"": ""regions"", ""expenditure_amount"": ""expenditure_amount"", ""unit_amount"": ""unit_amount""}",a5d05e136597e195af5ab6d78be6d895b227af2c7ebd5b19f3c59452234c87b0,d1c85596cecb41109d7f79ad07b35905,2026-10-12T14:58:10.107Z
03_regional_expenditure_gaa_2025.pdf,1,GAA (Updated),2025,DEPARTMENTS,All,Region II,119894181,thousand pesos,null,1wgAxpzoqU7VZtmwGC1IEi7cHvmk88MQR,03_regional_expenditure/03_regional_expenditure_gaa_2025.csv,/Volumes/pondolens/dev_bronze/government_budget/gdrive_landing/1wgAxpzoqU7VZtmwGC1IEi7cHvmk88MQR/3d392c835a11f16191582566fc20ddb49115f7e238a9e62dcb7e1c353ae51180.csv,2026-10-08T16:07:40.501Z,null,3d392c835a11f16191582566fc20ddb49115f7e238a9e62dcb7e1c353ae51180,"{""source_name"": ""source_name"", ""source_page"": ""source_page"", ""price_basis"": ""price_basis"", ""year"": ""year"", ""category"": ""category"", ""particulars"": ""particulars"", ""regions"": ""regions"", ""expenditure_amount"": ""expenditure_amount"", ""unit_amount"": ""unit_amount""}",a5d05e136597e195af5ab6d78be6d895b227af2c7ebd5b19f3c59452234c87b0,d1c85596cecb41109d7f79ad07b35905,2026-10-12T14:58:10.107Z
03_regional_expenditure_gaa_2025.pdf,1,GAA (Updated),2025,DEPARTMENTS,All,Region III,314090737,thousand pesos,null,1wgAxpzoqU7VZtmwGC1IEi7cHvmk88MQR,03_regional_expenditure/03_regional_expenditure_gaa_2025.csv,/Volumes/pondolens/dev_bronze/government_budget/gdrive_landing/1wgAxpzoqU7VZtmwGC1IEi7cHvmk88MQR/3d392c835a11f16191582566fc20ddb49115f7e238a9e62dcb7e1c353ae51180.csv,2026-10-08T16:07:40.501Z,null,3d392c835a11f16191582566fc20ddb49115f7e238a9e62dcb7e1c353ae51180,"{""source_name"": ""source_name"", ""source_page"": ""source_page"", ""price_basis"": ""price_basis"", ""year"": ""year"", ""category"": ""category"", ""particulars"": ""particulars"", ""regions"": ""regions"", ""expenditure_amount"": ""expenditure_amount"", ""unit_amount"": ""unit_amount""}",a5d05e136597e195af5ab6d78be6d895b227af2c7ebd5b19f3c59452234c87b0,d1c85596cecb41109d7f79ad07b35905,2026-10-12T14:58:10.107Z
03_regional_expenditure_gaa_2025.pdf,1,GAA (Updated),2025,DEPARTMENTS,All,Region IVA,272800469,thousand pesos,null,1wgAxpzoqU7VZtmwGC1IEi7cHvmk88MQR,03_regional_expenditure/0

In [0]:
%sql
-- List of all regions

SELECT
    DISTINCT(regions) as region_list
FROM pondolens.dev_bronze.regional_expenditure_gaa

region_list
Region II
Region III
CARAGA
BARMM
Region V
Region XII
Region X
Central Office
Region IVB
Region IX


In [0]:
%sql
SELECT COUNT(*)
FROM pondolens.dev_bronze.regional_expenditure_gaa;

COUNT(*)
4780


## 2. Data quality
Check required values, numeric/date parsing, rescued data, and repeated candidate keys. Empty duplicate results mean no repeated keys were found.


In [0]:
%sql
-- Null counts
SELECT
    COUNT(*) as total_rows,
    COUNT_IF(source_name IS NULL) as source_name_nulls,
    COUNT_IF(source_page IS NULL) as source_page_nulls,
    COUNT_IF(price_basis IS NULL) as price_basis_nulls,
    COUNT_IF(year IS NULL) as year_nulls,
    COUNT_IF(category IS NULL) as category_nulls,
    COUNT_IF(particulars IS NULL) as particulars_nulls,
    COUNT_IF(regions IS NULL) as regions_nulls,
    COUNT_IF(expenditure_amount IS NULL) as expenditure_amount_nulls,
    COUNT_IF(unit_amount IS NULL) as unit_amount_nulls
FROM
    pondolens.dev_bronze.regional_expenditure_gaa;

total_rows,source_name_nulls,source_page_nulls,price_basis_nulls,year_nulls,category_nulls,particulars_nulls,regions_nulls,expenditure_amount_nulls,unit_amount_nulls
4780,0,0,0,0,0,0,0,0,0


In [0]:
%sql
-- Duplicates
SELECT
    source_name,
    source_page,
    price_basis,
    year,
    category,
    particulars,
    regions,
    expenditure_amount,
    unit_amount,
    COUNT(*) AS duplicate_count
FROM pondolens.dev_bronze.regional_expenditure_gaa
GROUP BY
    source_name,
    source_page,
    price_basis,
    year,
    category,
    particulars,
    regions,
    expenditure_amount,
    unit_amount
HAVING COUNT(*) > 1;

source_name,source_page,price_basis,year,category,particulars,regions,expenditure_amount,unit_amount,duplicate_count


In [0]:
%sql
-- Check for special characters 

WITH column_values AS (
    SELECT
        source_name AS file_name,
        STACK(
            9,
            'source_name', CAST(source_name AS STRING),
            'source_page', CAST(source_page AS STRING),
            'price_basis', CAST(price_basis AS STRING),
            'year', CAST(year AS STRING),
            'category', CAST(category AS STRING),
            'particulars', CAST(particulars AS STRING),
            'regions', CAST(regions AS STRING),
            'expenditure_amount', CAST(expenditure_amount AS STRING),
            'unit_amount', CAST(unit_amount AS STRING)
        ) AS (column_name, column_value)
    FROM pondolens.dev_bronze.regional_expenditure_gaa
)
SELECT
    file_name,
    column_name,
    column_value,
    COUNT(*) AS occurrences
FROM column_values
WHERE column_value RLIKE '[^a-zA-Z0-9 ]'
GROUP BY file_name, column_name, column_value
ORDER BY column_name, occurrences DESC;

file_name,column_name,column_value,occurrences
03_regional_expenditure_gaa_2023.pdf,expenditure_amount,-,365
03_regional_expenditure_gaa_2024.pdf,expenditure_amount,-,344
03_regional_expenditure_gaa_2025.pdf,expenditure_amount,-,336
03_regional_expenditure_gaa_2022.pdf,expenditure_amount,-,333
03_regional_expenditure_gaa_2021.pdf,expenditure_amount,-,204
03_regional_expenditure_gaa_2021.pdf,particulars,Department of Foreign Affairs (DFA),20
03_regional_expenditure_gaa_2021.pdf,particulars,Joint Legislative-Executive Councils,20
03_regional_expenditure_gaa_2021.pdf,particulars,Commission on Elections (COMELEC),20
03_regional_expenditure_gaa_2021.pdf,particulars,Allocations to Local Government Units (ALGU),20
03_regional_expenditure_gaa_2021.pdf,particulars,Department of Agriculture (DA),20


## 3. Reconcile totals
- Geographical reconciliation: Does the sum of all regional amounts, Nationwide, and Central Office equal the reported Total?
- Particulars reconciliation: Does the sum of individual departments or special purpose funds equal the corresponding All subtotal?
- Grand total reconciliation: Does DEPARTMENTS + SPECIAL PURPOSE FUNDS equal GRAND TOTAL?

In [0]:
%sql
WITH reconciliation AS (
    SELECT
        year,
        category,
        particulars,

        SUM(
            CASE WHEN regions = 'Total'
            THEN TRY_CAST(expenditure_amount AS DECIMAL(20,2))
            ELSE 0 END
        ) AS reported_total,

        SUM(
            CASE WHEN regions <> 'Total'
            THEN TRY_CAST(expenditure_amount AS DECIMAL(20,2))
            ELSE 0 END
        ) AS calculated_total

    FROM pondolens.dev_bronze.regional_expenditure_gaa
    GROUP BY year, category, particulars
)

SELECT
    *,
    reported_total - calculated_total AS difference,
    CASE
        WHEN reported_total = calculated_total THEN 'MATCH'
        ELSE 'ISSUE'
    END AS status
FROM reconciliation
ORDER BY category, particulars;

year,category,particulars,reported_total,calculated_total,difference,status
2022,DEPARTMENTS,All,2964237560.00,2964237560.00,0.00,MATCH
2021,DEPARTMENTS,All,2713106666.00,2713106666.00,0.00,MATCH
2024,DEPARTMENTS,All,3598466387.00,3598466387.00,0.00,MATCH
2025,DEPARTMENTS,All,3780244169.00,3780244169.00,0.00,MATCH
2023,DEPARTMENTS,All,3234250422.00,3234250422.00,0.00,MATCH
2021,DEPARTMENTS,Civil Service Commission (CSC),1881434.00,1881434.00,0.00,MATCH
2024,DEPARTMENTS,Civil Service Commission (CSC),2342621.00,2342621.00,0.00,MATCH
2022,DEPARTMENTS,Civil Service Commission (CSC),1969425.00,1969425.00,0.00,MATCH
2023,DEPARTMENTS,Civil Service Commission (CSC),2154485.00,2154485.00,0.00,MATCH
2025,DEPARTMENTS,Civil Service Commission (CSC),2883229.00,2883229.00,0.00,MATCH


In [0]:
%sql
WITH reconciliation AS (
    SELECT
        year,
        category,
        regions,

        SUM(
            CASE WHEN particulars = 'All'
            THEN TRY_CAST(expenditure_amount AS DECIMAL(20,2))
            ELSE 0 END
        ) AS reported_total,

        SUM(
            CASE WHEN particulars <> 'All'
            THEN TRY_CAST(expenditure_amount AS DECIMAL(20,2))
            ELSE 0 END
        ) AS calculated_total

    FROM pondolens.dev_bronze.regional_expenditure_gaa
    WHERE category IN (
        'DEPARTMENTS',
        'SPECIAL PURPOSE FUNDS'
    )
    GROUP BY year, category, regions
)

SELECT
    *,
    reported_total - calculated_total AS difference,
    CASE
        WHEN reported_total = calculated_total THEN 'MATCH'
        ELSE 'ISSUE'
    END AS status
FROM reconciliation
ORDER BY category, regions;

year,category,regions,reported_total,calculated_total,difference,status
2021,DEPARTMENTS,BARMM,15135629.00,15135629.00,0.00,MATCH
2024,DEPARTMENTS,BARMM,21810984.00,21810984.00,0.00,MATCH
2023,DEPARTMENTS,BARMM,11663313.00,11663313.00,0.00,MATCH
2022,DEPARTMENTS,BARMM,17556117.00,17556117.00,0.00,MATCH
2025,DEPARTMENTS,BARMM,30268756.00,30268756.00,0.00,MATCH
2021,DEPARTMENTS,CARAGA,91286413.00,91286413.00,0.00,MATCH
2024,DEPARTMENTS,CARAGA,90291674.00,90291674.00,0.00,MATCH
2022,DEPARTMENTS,CARAGA,74669230.00,74669230.00,0.00,MATCH
2023,DEPARTMENTS,CARAGA,74486437.00,74486437.00,0.00,MATCH
2025,DEPARTMENTS,CARAGA,98414019.00,98414019.00,0.00,MATCH


In [0]:
%sql
WITH reconciliation AS (
    SELECT
        year,
        regions,

        SUM(
            CASE
                WHEN category = 'GRAND TOTAL'
                AND particulars = 'All'
                THEN TRY_CAST(expenditure_amount AS DECIMAL(20,2))
                ELSE 0
            END
        ) AS reported_grand_total,

        SUM(
            CASE
                WHEN category IN (
                    'DEPARTMENTS',
                    'SPECIAL PURPOSE FUNDS'
                )
                AND particulars = 'All'
                THEN TRY_CAST(expenditure_amount AS DECIMAL(20,2))
                ELSE 0
            END
        ) AS calculated_grand_total

    FROM pondolens.dev_bronze.regional_expenditure_gaa
    GROUP BY year, regions
)

SELECT
    *,
    reported_grand_total - calculated_grand_total AS difference,
    CASE
        WHEN reported_grand_total = calculated_grand_total
        THEN 'MATCH'
        ELSE 'ISSUE'
    END AS status
FROM reconciliation
ORDER BY regions;

year,regions,reported_grand_total,calculated_grand_total,difference,status
2021,BARMM,131371628.00,131371628.00,0.00,MATCH
2024,BARMM,149440406.00,149440406.00,0.00,MATCH
2023,BARMM,130285421.00,130285421.00,0.00,MATCH
2022,BARMM,143537923.00,143537923.00,0.00,MATCH
2025,BARMM,172504546.00,172504546.00,0.00,MATCH
2021,CARAGA,122719147.00,122719147.00,0.00,MATCH
2024,CARAGA,124270792.00,124270792.00,0.00,MATCH
2023,CARAGA,109329266.00,109329266.00,0.00,MATCH
2022,CARAGA,113226655.00,113226655.00,0.00,MATCH
2025,CARAGA,139115259.00,139115259.00,0.00,MATCH
